# M2-B1 — Audit de qualité et effet du nettoyage

Les **outils** sont fournis (rapport de qualité, comparaison des modèles). Les **constats**, les **décisions**
et leur **interprétation** sont à toi : réponds dans les cellules ✏️.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")
engine = create_engine(os.environ["DATABASE_URL"])

df = pd.read_sql(text("""
    SELECT h.code AS hotel, r.*
    FROM reservations AS r JOIN hotels AS h ON h.hotel_id = r.hotel_id
    ORDER BY r.arrival_date
"""), engine, parse_dates=["arrival_date", "reservation_status_date"])
df.shape

(78703, 35)

## Tâche 2 — L'audit

In [2]:
# Bornes « raisonnables » par colonne numérique : hypothèses de départ, à discuter
BORNES = {
    "lead_time": (0, 730),
    "adults": (1, 10),
    "children": (0, 10),
    "babies": (0, 5),
    "stays_in_weekend_nights": (0, 20),
    "stays_in_week_nights": (0, 50),
    "adr": (0, 1000),
}


def rapport_qualite(d: pd.DataFrame) -> pd.DataFrame:
    """Manquants et valeurs hors bornes, colonne par colonne."""
    lignes = []
    for col in d.columns:
        info = {"colonne": col, "type": str(d[col].dtype),
                "manquants": int(d[col].isna().sum()), "valeurs_distinctes": d[col].nunique()}
        if col in BORNES:
            bas, haut = BORNES[col]
            info["hors_bornes"] = int(((d[col] < bas) | (d[col] > haut)).sum())
        lignes.append(info)
    return pd.DataFrame(lignes).set_index("colonne")


rapport = rapport_qualite(df)
rapport[(rapport["manquants"] > 0) | (rapport.get("hors_bornes", 0) > 0)]

,type,manquants,valeurs_distinctes,hors_bornes
colonne,,,,
lead_time,int64,0,439,1.0
adults,int64,0,14,285.0
children,float64,4,5,0.0
babies,int64,0,5,2.0
country,object,405,159,NaN
agent,float64,10676,284,NaN
company,float64,74216,298,NaN
adr,float64,0,6985,1.0


In [3]:
# Doublons : réservations identiques sur toutes les colonnes, sauf l'identifiant
doublons = df.drop(columns="reservation_id").duplicated(keep="first")
print(doublons.sum(), "lignes en double")

22999 lignes en double


In [4]:
from IPython.display import display

COLS = ["hotel", "is_canceled", "reservation_status", "arrival_date", "lead_time",
        "stays_in_weekend_nights", "stays_in_week_nights", "adults", "children", "babies",
        "market_segment", "deposit_type", "customer_type", "adr"]


def montrer(nom, masque, n=5, cols=COLS):
    """Affiche le nombre de lignes, le taux d'annulation et quelques exemples."""
    print(f"{nom} : {int(masque.sum())} lignes, taux d'annulation {df.loc[masque, 'is_canceled'].mean():.1%}")
    display(df.loc[masque, cols].head(n))


montrer("children manquant", df["children"].isna())
montrer("country manquant", df["country"].isna(), n=3)
print("hôtel :", df.loc[df["country"].isna(), "hotel"].value_counts().to_dict())
montrer("adults hors bornes", (df["adults"] < 1) | (df["adults"] > 10))
montrer("babies hors bornes", df["babies"] > 5)
montrer("adr hors bornes", df["adr"] > 1000)
montrer("lead_time hors bornes", df["lead_time"] > 730)

children manquant : 4 lignes, taux d'annulation 100.0%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
2940,City Hotel,1,Canceled,2015-08-03,2,1,0,2,NaN,0,Undefined,No Deposit,Transient-Party,12.0
3189,City Hotel,1,Canceled,2015-08-05,1,0,2,2,NaN,0,Direct,No Deposit,Transient-Party,12.0
3201,City Hotel,1,Canceled,2015-08-05,1,0,2,3,NaN,0,Undefined,No Deposit,Transient-Party,18.0
4304,City Hotel,1,Canceled,2015-08-13,8,2,5,2,NaN,0,Online TA,No Deposit,Transient-Party,76.5


country manquant : 405 lignes, taux d'annulation 13.8%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
109,Resort Hotel,0,Check-Out,2015-07-01,118,4,10,1,0.0,0,Direct,No Deposit,Transient,62.0
112,Resort Hotel,0,Check-Out,2015-07-01,118,4,10,2,0.0,0,Direct,No Deposit,Transient,80.0
113,Resort Hotel,0,Check-Out,2015-07-01,4,0,1,1,0.0,0,Corporate,No Deposit,Transient,134.0


hôtel : {'Resort Hotel': 385, 'City Hotel': 20}
adults hors bornes : 285 lignes, taux d'annulation 27.7%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
1959,City Hotel,0,Check-Out,2015-07-23,132,2,5,0,0.0,0,Groups,No Deposit,Transient-Party,0.0
2251,City Hotel,0,Check-Out,2015-07-25,110,2,2,0,0.0,0,Offline TA/TO,No Deposit,Transient-Party,0.0
3748,City Hotel,0,Check-Out,2015-08-09,0,2,0,0,0.0,0,Online TA,No Deposit,Contract,0.0
3869,City Hotel,0,Check-Out,2015-08-10,1,1,1,0,3.0,0,Direct,No Deposit,Transient-Party,9.0
3905,City Hotel,0,Check-Out,2015-08-10,0,9,20,0,0.0,0,Online TA,No Deposit,Transient,0.0


... (2 outputs left out)

adr hors bornes : 1 lignes, taux d'annulation 100.0%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
31828,City Hotel,1,Canceled,2016-03-25,35,0,1,2,0.0,0,Offline TA/TO,Non Refund,Transient,5400.0


lead_time hors bornes : 1 lignes, taux d'annulation 0.0%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
80,Resort Hotel,0,Check-Out,2015-07-01,737,0,0,2,0.0,0,Direct,No Deposit,Transient,0.0


In [5]:
# agent manquant : canal de distribution de ces réservations
df.loc[df["agent"].isna(), "distribution_channel"].value_counts()

distribution_channel
Direct       5432
Corporate    3835
TA/TO        1405
Undefined       3
GDS             1
Name: count, dtype: int64

In [6]:
# Doublons : taille des paquets de lignes identiques, segments et types d'acompte concernés
cles = list(df.columns.drop("reservation_id"))
en_double = df.duplicated(subset=cles, keep=False)
paquets = df[en_double].groupby(cles, dropna=False).size()
print(f"{len(paquets)} paquets, {int(en_double.sum())} lignes, taille médiane {int(paquets.median())}, max {paquets.max()}")
print(f"taux d'annulation : doublons {df.loc[doublons, 'is_canceled'].mean():.1%} | ensemble {df['is_canceled'].mean():.1%}")
display(pd.DataFrame({
    "lignes": df["market_segment"].value_counts(),
    "part_en_double": doublons.groupby(df["market_segment"]).mean().round(3),
}).sort_values("part_en_double", ascending=False))
df.loc[doublons, "deposit_type"].value_counts()

5458 paquets, 28457 lignes, taille médiane 2, max 180
taux d'annulation : doublons 63.3% | ensemble 36.2%


,lignes,part_en_double
market_segment,,
Groups,13957,0.757
Offline TA/TO,18552,0.450
Corporate,3733,0.203
Online TA,33826,0.082
Direct,7977,0.063
Complementary,529,0.053
Aviation,127,0.047
Undefined,2,0.000


deposit_type
No Deposit    13188
Non Refund     9760
Refundable       51
Name: count, dtype: int64

### ✏️ Ce que le rapport montre

| Problème | Lignes | À quoi ça ressemble |
|---|---|---|
| `children` manquant | 4 | City Hotel, août 2015, toutes annulées. |
| `country` manquant | 405 | Surtout au Resort Hotel (385). Annulation 13,8 %. |
| `agent` manquant | 10 676 | Réservations directes ou entreprise, sans agence. Absence normale. Colonne non utilisée par le modèle. |
| `company` manquant | 74 216 | Réservations hors entreprise. Absence normale. Colonne non utilisée par le modèle. |
| `adults` hors bornes | 285 | 273 réservations à 0 adulte, 12 réservations de 20 à 55 adultes. |
| `babies` hors bornes | 2 | 9 et 10 bébés pour 1 et 2 adultes. |
| `adr` hors bornes | 1 | 5 400 € la nuit, 2 adultes, 1 nuit, annulée. |
| `lead_time` hors bornes | 1 | 737 jours, séjour de 0 nuit à 0 €. |
| Doublons | 22 999 | 28 457 lignes réparties en 5 458 paquets identiques, jusqu'à 180 lignes. |

Les doublons ne sont pas forcément des erreurs. La table n'a ni numéro de chambre ni nom de client : un groupe qui réserve
10 chambres identiques produit 10 lignes identiques. 75,7 % des lignes `Groups` sont en double.
Ils pèsent toutefois sur l'apprentissage : 63,3 % des doublons sont annulés, contre 36,2 % sur l'ensemble,
et 9 760 doublons sont des réservations `Non Refund`.

### ✏️ Ce que le rapport ne voit pas

Le rapport ne connaît que des bornes colonne par colonne. Cherche les **incohérences métier** : des combinaisons
de valeurs impossibles ou suspectes pour un hôtel. Au moins deux, chiffrées. Écris le code dans les cellules
ci-dessous.



#### 1. Séjours de 0 nuit

In [7]:
nuits = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
zero_nuit = nuits == 0
montrer("0 nuit", zero_nuit)
print("prix à 0 :", int((df.loc[zero_nuit, "adr"] == 0).sum()))
print("date de statut = date d'arrivée :",
      int((df.loc[zero_nuit, "reservation_status_date"] == df.loc[zero_nuit, "arrival_date"]).sum()))
print(df.loc[zero_nuit, "reservation_status"].value_counts().to_dict())

0 nuit : 571 lignes, taux d'annulation 4.2%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
79,Resort Hotel,0,Check-Out,2015-07-01,342,0,0,2,0.0,0,Direct,No Deposit,Transient,0.0
80,Resort Hotel,0,Check-Out,2015-07-01,737,0,0,2,0.0,0,Direct,No Deposit,Transient,0.0
449,Resort Hotel,0,Check-Out,2015-07-06,111,0,0,2,0.0,0,Online TA,No Deposit,Transient,0.0
450,Resort Hotel,0,Check-Out,2015-07-06,0,0,0,1,0.0,0,Direct,No Deposit,Transient,0.0
509,Resort Hotel,0,Check-Out,2015-07-07,8,0,0,2,0.0,0,Direct,No Deposit,Transient,0.0


prix à 0 : 571
date de statut = date d'arrivée : 571
{'Check-Out': 547, 'No-Show': 12, 'Canceled': 12}


571 réservations n'ont aucune nuit. Toutes ont un prix à 0 € et une date de statut égale à la date d'arrivée.
547 sont des `Check-Out`, le taux d'annulation est de 4,2 %.
Elles ressemblent à des passages à la journée ou à un autre type de prestation, pas à des séjours.

#### 2. Réservations sans adulte

In [8]:
sans_adulte = df["adults"] == 0
personnes = df["adults"] + df["children"].fillna(0) + df["babies"]
montrer("0 adulte", sans_adulte)
print("avec des enfants :", int((sans_adulte & (personnes > 0)).sum()), "| sans aucun client :", int((personnes == 0).sum()))
print("enfants :", df.loc[sans_adulte, "children"].value_counts().to_dict())
print("hôtel :", df.loc[sans_adulte, "hotel"].value_counts().to_dict(), "| prix à 0 :", int((df.loc[sans_adulte, "adr"] == 0).sum()))

0 adulte : 273 lignes, taux d'annulation 24.5%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
1959,City Hotel,0,Check-Out,2015-07-23,132,2,5,0,0.0,0,Groups,No Deposit,Transient-Party,0.0
2251,City Hotel,0,Check-Out,2015-07-25,110,2,2,0,0.0,0,Offline TA/TO,No Deposit,Transient-Party,0.0
3748,City Hotel,0,Check-Out,2015-08-09,0,2,0,0,0.0,0,Online TA,No Deposit,Contract,0.0
3869,City Hotel,0,Check-Out,2015-08-10,1,1,1,0,3.0,0,Direct,No Deposit,Transient-Party,9.0
3905,City Hotel,0,Check-Out,2015-08-10,0,9,20,0,0.0,0,Online TA,No Deposit,Transient,0.0


avec des enfants : 160 | sans aucun client : 113
enfants : {2.0: 150, 0.0: 113, 3.0: 8, 1.0: 2}
hôtel : {'City Hotel': 265, 'Resort Hotel': 8} | prix à 0 : 97


273 réservations n'ont aucun adulte, dont 265 au City Hotel. 160 n'ont que des enfants, le plus souvent 2.
113 n'ont aucun client. 97 ont aussi un prix à 0 €.
Des enfants seuls sont peu plausibles ; une réservation sans client est impossible.
Hypothèse : les adultes ont été saisis dans `children`.

#### 3. Prix à 0 €

In [9]:
prix_nul = df["adr"] == 0
offert = df["market_segment"] == "Complementary"
print("prix à 0 par segment :", df.loc[prix_nul, "market_segment"].value_counts().to_dict())
prix_nul_suspect = prix_nul & ~offert & ~zero_nuit
montrer("prix à 0, hors séjours offerts et hors 0 nuit", prix_nul_suspect)
print(df.loc[prix_nul_suspect, "customer_type"].value_counts().to_dict())
print("séjours offerts avec un prix > 0 :", int((offert & (df["adr"] > 0)).sum()))

prix à 0 par segment : {'Complementary': 479, 'Offline TA/TO': 295, 'Online TA': 287, 'Groups': 210, 'Direct': 165, 'Corporate': 66, 'Aviation': 2}
prix à 0, hors séjours offerts et hors 0 nuit : 471 lignes, taux d'annulation 17.8%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
0,City Hotel,0,Check-Out,2015-07-01,6,0,2,1,0.0,0,Offline TA/TO,No Deposit,Transient,0.0
579,City Hotel,0,Check-Out,2015-07-08,264,0,2,1,0.0,0,Groups,No Deposit,Transient-Party,0.0
580,City Hotel,0,Check-Out,2015-07-08,264,0,2,1,0.0,0,Groups,No Deposit,Transient-Party,0.0
1026,City Hotel,1,Canceled,2015-07-15,0,0,1,2,0.0,0,Direct,No Deposit,Transient,0.0
1031,City Hotel,0,Check-Out,2015-07-15,0,2,5,2,0.0,0,Direct,No Deposit,Transient,0.0


{'Transient-Party': 298, 'Transient': 149, 'Group': 19, 'Contract': 5}
séjours offerts avec un prix > 0 : 50


1 504 réservations sont à 0 €. 479 sont des séjours offerts (`Complementary`) : le prix nul y est normal.
Les 571 séjours de 0 nuit en font partie.
Restent 471 réservations payantes d'au moins une nuit à 0 €, surtout en `Transient-Party` (298), annulées à 17,8 %.
À l'inverse, 50 séjours offerts ont un prix supérieur à 0.
Hors séjours offerts, un prix nul peut être une erreur ou un forfait facturé ailleurs (groupe, agence) : à demander au client.

#### 4. Acomptes non remboursables annulés

In [10]:
non_remb = df["deposit_type"] == "Non Refund"
display(df.groupby("deposit_type").agg(
    lignes=("is_canceled", "size"), taux_annulation=("is_canceled", "mean"),
    delai_median=("lead_time", "median"), demandes_speciales=("total_of_special_requests", "mean"),
).round(3))
print(f"pays PRT : {(df.loc[non_remb, 'country'] == 'PRT').mean():.1%}")
print("segments :", df.loc[non_remb, "market_segment"].value_counts().head(3).to_dict())
print("lignes dans un paquet de doublons :", int(en_double[non_remb].sum()))
print(f"previous_cancellations > 0 : Non Refund {(df.loc[non_remb, 'previous_cancellations'] > 0).mean():.1%}"
      f" | autres {(df.loc[~non_remb, 'previous_cancellations'] > 0).mean():.1%}")
print("dates d'annulation les plus fréquentes :",
      df.loc[non_remb & (df["is_canceled"] == 1), "reservation_status_date"].dt.date.value_counts().head(3).to_dict())
montrer("Non Refund non annulées", non_remb & (df["is_canceled"] == 0))

,lignes,taux_annulation,delai_median,demandes_speciales
deposit_type,,,,
No Deposit,68107,0.266,50.0,0.597
Non Refund,10461,0.992,238.0,0.002
Refundable,135,0.074,169.0,0.015


pays PRT : 96.2%
segments : {'Groups': 6395, 'Offline TA/TO': 3856, 'Corporate': 146}
lignes dans un paquet de doublons : 10325
previous_cancellations > 0 : Non Refund 34.9% | autres 3.8%
dates d'annulation les plus fréquentes : {datetime.date(2015, 10, 21): 1136, datetime.date(2015, 1, 1): 540, datetime.date(2016, 1, 18): 511}
Non Refund non annulées : 85 lignes, taux d'annulation 0.0%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
12219,City Hotel,0,Check-Out,2015-10-03,1,0,1,1,0.0,0,Direct,Non Refund,Transient-Party,145.0
13993,Resort Hotel,0,Check-Out,2015-10-12,7,1,3,2,0.0,0,Corporate,Non Refund,Transient-Party,109.0
13994,Resort Hotel,0,Check-Out,2015-10-12,7,1,3,2,0.0,0,Corporate,Non Refund,Transient-Party,109.0
13995,Resort Hotel,0,Check-Out,2015-10-12,7,1,3,2,0.0,0,Corporate,Non Refund,Transient-Party,109.0
13996,Resort Hotel,0,Check-Out,2015-10-12,7,1,3,2,0.0,0,Corporate,Non Refund,Transient-Party,109.0


10 461 réservations ont un acompte non remboursable. 99,2 % sont annulées, contre 26,6 % sans acompte.
Un acompte perdu devrait dissuader d'annuler, pas l'inverse.
Profil : 96 % de clients portugais, segments `Groups` et `Offline TA/TO`, délai médian de 238 jours,
presque aucune demande spéciale. 10 325 de ces lignes sont dans des paquets de doublons.
1 136 annulations portent la même date, le 21/10/2015.
Ces lignes ressemblent à des blocs de chambres posés par des agences puis libérés, plutôt qu'à des clients ayant payé.
Pour le modèle, `deposit_type = Non Refund` équivaut presque à une annulation certaine.

#### 5. Valeurs extrêmes

In [11]:
mois = df["arrival_date"].dt.to_period("M")
print("prix médian City Hotel, mars 2016 :",
      df.loc[(df["hotel"] == "City Hotel") & (mois == "2016-03"), "adr"].median())
montrer("20 adultes ou plus", df["adults"] >= 20, n=12)

prix médian City Hotel, mars 2016 : 84.33
20 adultes ou plus : 12 lignes, taux d'annulation 100.0%


,hotel,is_canceled,reservation_status,arrival_date,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,market_segment,deposit_type,customer_type,adr
6989,Resort Hotel,1,Canceled,2015-09-03,304,0,3,40,0.0,0,Direct,No Deposit,Group,0.0
7435,Resort Hotel,1,Canceled,2015-09-05,333,2,5,26,0.0,0,Offline TA/TO,No Deposit,Group,0.0
7716,Resort Hotel,1,Canceled,2015-09-07,336,1,2,50,0.0,0,Direct,No Deposit,Group,0.0
8554,Resort Hotel,1,Canceled,2015-09-12,340,2,5,26,0.0,0,Offline TA/TO,No Deposit,Group,0.0
9921,Resort Hotel,1,Canceled,2015-09-19,347,2,5,26,0.0,0,Offline TA/TO,No Deposit,Group,0.0
10286,Resort Hotel,1,Canceled,2015-09-21,349,1,3,27,0.0,0,Direct,No Deposit,Group,0.0
10769,Resort Hotel,1,Canceled,2015-09-24,352,1,3,27,0.0,0,Direct,No Deposit,Group,0.0
11224,Resort Hotel,1,Canceled,2015-09-26,354,2,5,26,0.0,0,Offline TA/TO,No Deposit,Group,0.0
12261,Resort Hotel,1,Canceled,2015-10-03,361,2,5,26,0.0,0,Offline TA/TO,No Deposit,Group,0.0
12428,Resort Hotel,1,Canceled,2015-10-04,338,2,0,55,0.0,0,Direct,No Deposit,Group,0.0


Le prix de 5 400 € pour une nuit à 2 adultes se compare à un prix médian de 84 € ce mois-là : erreur de saisie probable.
Les 12 réservations de 20 à 55 adultes sont au Resort Hotel, en septembre-octobre 2015, de type `Group`, à 0 €,
toutes annulées. Ce sont des blocs de groupe, pas des chambres.
Les 2 réservations à 9 et 10 bébés sont des erreurs de saisie probables.

#### 6. Codes incohérents

In [12]:
display(pd.crosstab(df["meal"], df["hotel"]))
pays = df.groupby("country")["is_canceled"].agg(lignes="size", taux_annulation="mean").round(3)
display(pays.reindex(["CN", "CHN", "CAN", "USA"]))
print("Undefined :", {c: int((df[c] == "Undefined").sum()) for c in ["market_segment", "distribution_channel"]})

hotel,City Hotel,Resort Hotel
meal,,
BB,41536,20250
FB,41,662
HB,4820,5166
SC,5425,47
Undefined,0,756


,lignes,taux_annulation
country,,
CN,641.0,0.164
CHN,565.0,0.444
CAN,NaN,NaN
USA,1143.0,0.188


Undefined : {'market_segment': 2, 'distribution_channel': 5}


`meal = Undefined` (756) n'existe qu'au Resort Hotel, où `SC` (sans repas) est presque absent (47).
C'est probablement la même formule, codée différemment selon l'hôtel. `meal` n'est pas utilisée par le modèle.
`CN` (641) est le seul code pays à 2 lettres. `CAN` (Canada) est absent de la base.
`CN` est annulé à 16,4 %, `CHN` (Chine, 565) à 44,4 % : `CN` désigne peut-être le Canada.
`country` est utilisée par le modèle.
2 lignes ont `market_segment = Undefined` et 5 `distribution_channel = Undefined`.

#### 7. Contrôles sans anomalie

In [13]:
display(pd.crosstab(df["is_canceled"], df["reservation_status"]))
depart = df["arrival_date"] + pd.to_timedelta(nuits, unit="D")
annule = df["reservation_status"] == "Canceled"
date_resa = df["arrival_date"] - pd.to_timedelta(df["lead_time"], unit="D")
print("Check-Out avant le départ prévu :",
      int(((df["reservation_status"] == "Check-Out") & (df["reservation_status_date"] < depart)).sum()))
print("annulations datées après l'arrivée :", int((annule & (df["reservation_status_date"] > df["arrival_date"])).sum()))
print("annulations datées avant la réservation :", int((annule & (df["reservation_status_date"] < date_resa)).sum()))

reservation_status,Canceled,Check-Out,No-Show
is_canceled,,,
0,0,50224,0
1,27620,0,859


Check-Out avant le départ prévu : 23
annulations datées après l'arrivée : 0
annulations datées avant la réservation : 0


`is_canceled` est cohérent avec `reservation_status`. Aucune annulation n'est datée après l'arrivée
ni avant la date de réservation. 23 `Check-Out` ont lieu avant le départ prévu : départs anticipés plausibles.

#### Synthèse

In [14]:
coupure = df["arrival_date"].max() - pd.DateOffset(months=4)   # même découpage que train.py
dans_test = df["arrival_date"] > coupure
anomalies = {
    "doublons": doublons,
    "Non Refund": non_remb,
    "prix 0 hors offerts et hors 0 nuit": prix_nul_suspect,
    "0 nuit": zero_nuit,
    "country manquant": df["country"].isna(),
    "0 adulte": sans_adulte,
    "valeurs extrêmes": (df["adr"] > 1000) | (df["babies"] > 5) | (df["adults"] >= 20),
    "children manquant": df["children"].isna(),
}
pd.DataFrame({nom: {
    "lignes": int(m.sum()), "train": int((m & ~dans_test).sum()), "test": int((m & dans_test).sum()),
    "taux_annulation": round(df.loc[m, "is_canceled"].mean(), 3),
} for nom, m in anomalies.items()}).T.astype({"lignes": int, "train": int, "test": int})

,lignes,train,test,taux_annulation
doublons,22999,17652,5347,0.633
Non Refund,10461,8035,2426,0.992
prix 0 hors offerts et hors 0 nuit,471,425,46,0.178
0 nuit,571,420,151,0.042
country manquant,405,313,92,0.138
0 adulte,273,184,89,0.245
valeurs extrêmes,15,15,0,0.867
children manquant,4,4,0,1.000


Les anomalies se recouvrent : les séjours de 0 nuit sont tous à 0 €, la plupart des `Non Refund` sont des doublons.
Le jeu de test (25 % des lignes) contient aussi la plupart de ces anomalies.
Les valeurs extrêmes et les `children` manquants sont tous dans l'entraînement.

## Tâche 3 — Les décisions

| Anomalie | Lignes | Décision (supprimer / corriger / imputer / garder et signaler) | Justification | Question au client ? |
|---|---|---|---|---|
| `children` manquant | 4 | imputer à 0 (règle d'exemple) | Cas le plus fréquent. | Non |
| Aucun client | 113 | supprimer | Réservation impossible. | Non |
| Séjour de 0 nuit | 571 | supprimer | Pas de séjour à annuler. | Oui |
| Prix de 5 400 € | 1 | supprimer | Erreur de saisie. | Non |
| 20 adultes ou plus | 12 | supprimer | Blocs de groupe, pas des chambres. | Non |
| `Non Refund` annulé à 99 % | 10 461 | garder et signaler | Sens du code incertain. | Oui |
| Doublons | 22 999 | garder et signaler | Pas de numéro de chambre. | Oui |
| Prix à 0 € hors séjours offerts | 471 | garder et signaler | Erreur ou forfait : inconnu. | Oui |
| Enfants sans adulte | 160 | garder et signaler | Saisie douteuse, non vérifiée. | Oui |
| `country = CN` | 641 | garder et signaler | Canada ou Chine : inconnu. | Oui |
| `country` manquant | 405 | garder et signaler | Le pipeline met `Unknown`. | Non |
| `meal = Undefined`, 9 ou 10 bébés | 756 / 2 | garder et signaler | Colonnes non utilisées par le modèle. | Non |
| Écarts mineurs : codes `Undefined`, séjours offerts payants, départs anticipés | 7 / 50 / 23 | garder | Peu de lignes. | Non |

Les suppressions retirent 661 lignes (0,8 %), dont 172 dans le jeu de test.

### Questions mises de côté

Par ordre d'importance, avec la décision provisoire.

1. Que signifie `Non Refund` ? Ces réservations ont-elles été payées ? Provisoire : gardées.
2. Une ligne correspond-elle à une chambre ? Provisoire : doublons gardés.
3. Pourquoi des séjours payants sont-ils à 0 € ? Provisoire : gardés.
4. Une réservation d'enfants sans adulte est-elle possible ? Provisoire : gardées.
5. Le code pays `CN` désigne-t-il le Canada ou la Chine ? Provisoire : gardé tel quel.
6. Les séjours de 0 nuit sont-ils des passages à la journée ? Provisoire : supprimés.

## Tâche 6 — Comparer v2.0.0 et v2.1.0

À faire **après** avoir branché le nettoyage et produit la v2.1.0 avec `python train.py`.
La fonction de comparaison est fournie : c'est le **choix du jeu de test** qui t'appartient.

In [15]:
import joblib
from sklearn.metrics import f1_score, precision_score, recall_score

import train as T
from features import INPUT_FEATURES


def comparer(modeles: dict, X: pd.DataFrame, y: pd.Series) -> pd.DataFrame:
    lignes = []
    for nom, modele in modeles.items():
        pred = modele.predict(X)
        lignes.append({"modèle": nom, "f1": f1_score(y, pred), "rappel": recall_score(y, pred),
                       "précision": precision_score(y, pred), "n": len(y)})
    return pd.DataFrame(lignes).set_index("modèle").round(3)


modeles = {
    "v2.0.0 (données brutes)": joblib.load(ROOT / "model" / "cancellation_model_v2.0.0.joblib"),
    # "v2.1.0 (données nettoyées)": joblib.load(ROOT / "model" / "cancellation_model_v2.1.0.joblib"),
}

brut = T.load_data(engine)
_, test_brut = T.split_train_test(brut)
comparer(modeles, test_brut[INPUT_FEATURES], test_brut[T.TARGET])

,f1,rappel,précision,n
modèle,,,,
v2.0.0 (données brutes),0.734,0.754,0.715,19911


In [16]:
# TODO : construire le test nettoyé, et comparer les deux modèles dessus aussi

### ✏️ Interprétation

1. Le score monte, baisse ou reste stable : pourquoi ? Une baisse est-elle forcément une mauvaise nouvelle ?
2. Quelles réservations ont été exclues, et que fera le modèle s'il en reçoit de semblables en production ?
3. Faut-il appliquer le même nettoyage aux réservations que recevra l'API ?
4. Lequel des deux jeux de test, brut ou nettoyé, ressemble le plus à la production ? Lequel retiens-tu pour décider ?

*…*